# Przestrzenie stanów – interaktywnie
**Wprowadzenie do AI · laboratorium 2**

Plan:
0. **Wprowadzenie** – jak sformułować problem i narysować przestrzeń stanów *na kartce* (na prostym przykładzie windy),
1. **Świat odkurzacza** i 2. **Pojemniki z wodą** – to samo, ale przestrzeń stanów generuje program, a Ty zmieniasz parametry i obserwujesz.

> **Jak uruchomić:** uruchom komórki po kolei (`Shift+Enter`). Pierwsza komórka (ukryta) sama pobierze potrzebny kod – nic nie trzeba wgrywać.

In [ ]:
#@title ⚙️ Przygotowanie – uruchom tę komórkę (kod ukryty)
import os, sys, subprocess
REPO = "https://github.com/BartekDaniluk/kognitywistyka_intro_to_AI"
LAB = "lab02_stany"
if "google.colab" in sys.modules:
    root = "/content/kognitywistyka_intro_to_AI"
    if not os.path.exists(root):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, root], check=True)
    else:
        subprocess.run(["git", "-C", root, "pull", "-q"], check=False)
    sys.path.insert(0, os.path.join(root, LAB))
%matplotlib inline
import lab_states as ls
ls.set_language("pl")   # "en" = interfejs po angielsku
print("Gotowe ✔")

## 0. Wprowadzenie: przestrzeń stanów na kartce

Wiele problemów – łamigłówki, planowanie trasy, sprzątanie – da się opisać w ten sam sposób: jesteśmy w jakiejś **sytuacji**,
możemy wykonać jedną z kilku **czynności**, każda czynność zmienia sytuację, a chcemy dojść do sytuacji **docelowej**.
Zanim ktokolwiek – człowiek albo komputer – zacznie szukać rozwiązania, problem trzeba **sformułować**.
Zrobimy to krok po kroku na prostym przykładzie.

> 🛗 **Przykład: winda.** Budynek ma 3 kondygnacje: parter (0), 1. i 2. piętro. Winda stoi na parterze z **otwartymi** drzwiami.
> Chcemy, żeby dojechała na 2. piętro i otworzyła drzwi. Winda może jechać **tylko przy zamkniętych drzwiach**.

### Krok 1. Stan – „zdjęcie” sytuacji

**Stan** to opis wszystkiego, co jest istotne dla problemu w danej chwili. Zwykle zapisujemy go jako **krotkę** – uporządkowany ciąg zmiennych.

Co jest istotne dla windy? Piętro, na którym stoi, i to, czy drzwi są otwarte. Co **pomijamy**? Kolor kabiny, pasażerów, godzinę…
To jest **abstrakcja** – świadomie wyrzucamy szczegóły, które nie wpływają na rozwiązanie.

Stan windy zapiszemy jako **(P, D)**, gdzie:
* P ∈ {0, 1, 2} – piętro,
* D ∈ {o, z} – drzwi otwarte / zamknięte.

Np. **(1, z)** = „winda na 1. piętrze, drzwi zamknięte”.

Dobra reprezentacja stanu jest:
* **jednoznaczna** – jeden zapis oznacza dokładnie jedną sytuację,
* **zupełna** – zawiera wszystko, czego potrzeba, żeby wiedzieć, co wolno zrobić dalej i czy osiągnęliśmy cel.

✏️ *Pytanie:* czy wystarczyłoby zapisywać samo piętro P? Czego wtedy nie dałoby się rozstrzygnąć?

### Krok 2. Przestrzeń stanów – ile jest stanów?

**Przestrzeń stanów** to zbiór wszystkich możliwych stanów. Jeśli stan jest krotką zmiennych, liczbę stanów dostajemy,
**mnożąc** liczby możliwych wartości poszczególnych zmiennych:

$$|S| = |\{0, 1, 2\}| \cdot |\{o, z\}| = 3 \cdot 2 = 6$$

Wypisane: (0, z), (0, o), (1, z), (1, o), (2, z), (2, o).

Uwaga: nie każdy zapis musi mieć sens. W innych problemach część stanów jest **niedozwolona** (np. gdy kanibali na brzegu jest więcej
niż misjonarzy) albo **nieosiągalna** – nie da się do nich dojść ze stanu początkowego. Zobaczysz to przy pojemnikach.

### Krok 3. Stan początkowy i cel

* **Stan początkowy:** (0, o)
* **Test celu:** P = 2 i D = o, czyli stan docelowy **(2, o)**.

Cel nie musi być jednym stanem – może być warunkiem, który spełnia wiele stanów (np. „drzwi otwarte, piętro dowolne”).

### Krok 4. Akcje i model przejścia – reguły

**Akcja** to czynność, którą można wykonać. **Model przejścia** mówi, do jakiego stanu prowadzi dana akcja wykonana w danym stanie.
Na kartce zapisujemy to jako ponumerowane **reguły**:

> (stan | warunek) → nowy stan

Warunek mówi, **kiedy** wolno zastosować regułę. Dla windy:

1. `(P, z | P < 2) → (P+1, z)` – jedź w górę
2. `(P, z | P > 0) → (P−1, z)` – jedź w dół
3. `(P, z) → (P, o)` – otwórz drzwi
4. `(P, o) → (P, z)` – zamknij drzwi

Zwróć uwagę, że reguły 1 i 2 mają po lewej stronie **z** – w ten sposób zapisaliśmy, że jechać można tylko przy zamkniętych drzwiach.
Reguły są **ogólne** (używają zmiennych P, D) – nie trzeba wypisywać osobnej reguły dla każdego piętra.

### Krok 5. Koszt akcji

Przyjmijmy, że każda akcja kosztuje **1** – wtedy najtańsze rozwiązanie to po prostu najkrótsze (najmniej czynności).
Gdyby np. jazda kosztowała więcej niż otwieranie drzwi, zapisalibyśmy to właśnie tutaj.

### Krok 6. Tabela przejść

Teraz systematycznie: dla każdego stanu (wiersz) i każdej reguły (kolumna) sprawdzamy warunek.
Jeśli jest spełniony – wpisujemy stan wynikowy, jeśli nie – kreskę.

| stan | 1 (w górę) | 2 (w dół) | 3 (otwórz) | 4 (zamknij) |
|---|---|---|---|---|
| (0, z) | (1, z) | — | (0, o) | — |
| (0, o) | — | — | — | (0, z) |
| (1, z) | (2, z) | (0, z) | (1, o) | — |
| (1, o) | — | — | — | (1, z) |
| (2, z) | — | (1, z) | (2, o) | — |
| (2, o) | — | — | — | (2, z) |

### Krok 7. Graf stanów

Stany to **węzły**, przejścia to **strzałki** podpisane numerem reguły. Na kartce najwygodniej rysować graf **od stanu początkowego**:

1. Zapisz stan początkowy.
2. Weź stan, z którego jeszcze nie „wychodziłeś”. Sprawdź po kolei wszystkie reguły; dla każdej, której warunek jest spełniony,
   narysuj strzałkę do stanu wynikowego (jeśli tego stanu jeszcze nie ma na kartce – dopisz go).
3. Powtarzaj krok 2, aż nie pojawiają się nowe stany.

Dzięki temu na kartce lądują tylko stany **osiągalne** ze stanu początkowego. Dla windy wychodzi:

```
 start
   ↓
 (0,o)           (1,o)           (2,o)  ← cel
 4 │ ▲ 3         4 │ ▲ 3         4 │ ▲ 3
   ▼ │             ▼ │             ▼ │
 (0,z) ───1───▶ (1,z) ───1───▶ (2,z)
       ◀───2───       ◀───2───
```

### Krok 8. Rozwiązanie

**Rozwiązanie** to ścieżka w grafie od stanu początkowego do stanu docelowego, zapisana jako ciąg numerów reguł:

> (0, o) –4→ (0, z) –1→ (1, z) –1→ (2, z) –3→ (2, o), czyli **4 – 1 – 1 – 3** (4 kroki, koszt 4).

Rozwiązanie **optymalne** to takie o najmniejszym koszcie.

### A teraz to samo – komputerem

Program robi dokładnie to, co Ty na kartce: dostaje definicję (stany, reguły z warunkami, start, cel, koszt),
wypełnia tabelę przejść, rysuje graf zaczynając od stanu początkowego i szuka najtańszej ścieżki.
Uruchom komórkę i porównaj wynik z rysunkiem i tabelą powyżej.

In [ ]:
winda = ls.Elevator(floors=3, initial=(0, 'o'), goal=(2, 'o'))   # None w celu = „dowolnie”
ls.show(winda)
ls.show_table(winda)

### ✏️ Sprawdź się
Najpierw na kartce, potem sprawdź, zmieniając liczby w komórce powyżej.

1. Budynek ma **4** kondygnacje (`floors=4`). Ile jest stanów? Która reguła się zmienia?
2. Cel: „drzwi otwarte, piętro dowolne” (`goal=(None, 'o')`). Ile jest stanów docelowych? Jakie jest rozwiązanie – i dlaczego takie?
3. *(tylko na kartce)* Dodaj do stanu informację, czy w windzie jest pasażer, oraz akcje „wsiądź” i „wysiądź” (możliwe tylko przy otwartych drzwiach).
   Jak wygląda teraz stan? Ile jest stanów? Zapisz nowe reguły w postaci `(stan | warunek) → nowy stan`.

### Podsumowanie

Każdy problem przeszukiwania definiujemy przez pięć składników:

| Składnik | Pytanie | Winda |
|---|---|---|
| **stany** | jak jednoznacznie i zupełnie zapisać sytuację? | (P, D) |
| **stan początkowy** | od czego zaczynamy? | (0, o) |
| **test celu** | kiedy kończymy? | (2, o) |
| **akcje + model przejścia** | co wolno zrobić w danym stanie i do czego to prowadzi? | reguły 1–4 |
| **koszt akcji** | ile kosztuje każdy krok? | 1 |

W kolejnych częściach te same składniki zobaczysz w zakładce *Definicja formalna* – dla odkurzacza i pojemników.

## 1. Świat odkurzacza

Robot-odkurzacz porusza się po rzędzie pól (A, B, …). Każde pole może być brudne (ś) lub czyste (c).
Akcje: **W lewo**, **W prawo**, **Odkurzaj** (opcjonalnie **Nic nie rób**). Cel: wszystkie pola czyste.

**Jak korzystać:**
* ustaw parametry na górze – graf, definicja i tabela przejść przeliczą się same,
* w zakładce *Graf + symulacja* klikaj przyciski akcji i obserwuj, jak „wędrujesz” po grafie (twoja ścieżka – kolor pomarańczowy),
* zaznacz *Pokaż rozwiązanie optymalne*, żeby sprawdzić, czy znalazłeś najkrótszą/najtańszą drogę.

In [ ]:
ls.vacuum_app()

### Zadania
1. Ile stanów ma świat z **2** polami? Z **3**? Z **4**? Najpierw policz na kartce, potem sprawdź w zakładce *Definicja formalna*. Podaj wzór dla *n* pól.
2. Ustaw *Maks. brudnych* = 1. Jak zmienia się liczba stanów? Dlaczego graf ma tylko „dwie kolumny”?
3. Dla 3 pól zacznij od pola **B**, wszystkie pola brudne. Znajdź rozwiązanie samodzielnie klikając akcje. Porównaj z rozwiązaniem optymalnym.
4. Ustaw *Koszt ruchu* = 3, *Koszt odkurzania* = 1. Czy rozwiązanie optymalne się zmieniło? A gdy koszt ruchu = 0?
5. Zaznacz *Pokaż też stany nieosiągalne* przy starcie z dwoma czystymi polami. Które stany są nieosiągalne i dlaczego?
6. Które akcje tworzą **pętle** (nie zmieniają stanu)? Czy mają sens w rozwiązaniu?

## 2. Pojemniki z wodą

Masz dwa naczynia: **A** (4 l) i **B** (3 l), bez podziałki. Możesz napełniać je z kranu, wylewać wodę do zlewu
i przelewać wodę z jednego do drugiego. Zadanie: w naczyniu A mają się znaleźć **dokładnie 2 litry**, B ma być puste.

Stan zapisujemy jako **(X, Y)** – ilość wody w A i w B. Akcje to reguły **1–8** (zakładka *Definicja formalna* pokazuje
warunki ich stosowania). W grafie liczby na krawędziach to numery reguł, a rozwiązanie zapisujemy jako ciąg numerów, np. `2 – 8 – …`.
Węzły ułożone są w siatce: oś pozioma = woda w A, oś pionowa = woda w B.

In [ ]:
ls.water_jugs_app()

### Zadania
1. Rozwiąż problem 4 l / 3 l samodzielnie (przyciski). Ile kroków potrzebowałeś? Porównaj z rozwiązaniem optymalnym.
2. Znajdź **drugie**, inne rozwiązanie. Jakie są dwie „strategie” (od której reguły się zaczyna)?
3. Zaznacz *Pokaż też stany nieosiągalne*. Które stany są nieosiągalne? Co je łączy (spójrz na położenie w siatce)?
4. Zmień koszt na *litry przelanej wody*. Czy najtańsze rozwiązanie to nadal najkrótsze?
5. Pojemniki **5 l i 3 l**, cel: 4 l w A (B dowolnie). Rozwiąż.
6. Pojemniki **6 l i 4 l**, cel: 1 l w A. Co się dzieje? Spróbuj innych par pojemności – kiedy problem nie ma rozwiązania? *(wskazówka: NWD)*
7. Które reguły są potrzebne w rozwiązaniu, a których nigdy nie użyto?

## 3. Dla ciekawskich: bez przycisków

Te same problemy można tworzyć „w kodzie” – wystarczy zmienić liczby w nawiasach i uruchomić komórkę.

In [ ]:
p = ls.WaterJugs(cap_a=4, cap_b=3, initial=(0, 0), goal=(2, 0))   # None = „dowolnie”
ls.show(p)            # definicja + graf z rozwiązaniem
ls.show_table(p)      # tabela przejść

In [ ]:
v = ls.VacuumWorld(n=3, initial=(1, (1, 0, 1)))   # robot na B; A brudne, B czyste, C brudne
kroki, koszt = ls.solve(v)
for s, a, s2 in kroki:
    print(v.state_label(s), '--', v.action_name(a), '->', v.state_label(s2))
print('koszt:', koszt)